# Day 08 — Functions + Reusable Code
**Estimated time:** 60–75 min
**Datasets:** `materials_inventory.csv`, `sales_orders.csv`

## Learning Objectives
- Write clean, reusable helper functions for data wrangling tasks
- Apply functions with `.apply()`, `.map()`, and vectorized alternatives — know which to use when
- Build a parameterized data quality report function

In [ ]:
import pandas as pd
import numpy as np
from typing import Optional

DATA_DIR = "../datasets"

inv = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
inv["LABST"] = pd.to_numeric(inv["LABST"], errors="coerce")
inv["STPRS"] = pd.to_numeric(inv["STPRS"], errors="coerce")
inv["LAST_MOVEMENT_DATE"] = pd.to_datetime(inv["LAST_MOVEMENT_DATE"], errors="coerce")

sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
sales["ERDAT"] = pd.to_datetime(sales["ERDAT"], errors="coerce")
sales["NETWR"] = pd.to_numeric(sales["NETWR"], errors="coerce")

print("Loaded. inv:", inv.shape, "| sales:", sales.shape)

In [ ]:
# -- 1. Helper function: standardize SAP text fields --

def clean_sap_text(value, pad_zeros: Optional[int] = None) -> str:
    # Standardize a SAP text/key field.
    # Strips whitespace, uppercases, collapses internal spaces.
    # Optionally zero-pads to pad_zeros length.
    if pd.isna(value):
        return ""
    cleaned = " ".join(str(value).strip().upper().split())
    if pad_zeros is not None:
        cleaned = cleaned.zfill(pad_zeros)
    return cleaned

# Test
test_vals = ["  bearing assembly  ", "GEAR BOX", None, "valve-12 "]
print([clean_sap_text(v) for v in test_vals])
print([clean_sap_text(v, pad_zeros=18) for v in ["100","200000","  50 "]])

In [ ]:
# -- 2. .map() vs .apply() — when to use each --
#
# .map()   -> element-wise on a Series. Pure transformation. Fast.
# .apply() -> row or column-wise on DataFrame. More flexible, slower.
#             Use when you need multiple columns per row.

# FAST: .map() with a function on a single Series
inv["MAKTX_CLEAN"] = inv["MAKTX"].map(clean_sap_text)

# .apply() is justified when you need multiple columns
def classify_material(row):
    # Classify material risk based on stock and last movement
    if row["LABST"] == 0:
        return "Zero Stock"
    days = (pd.Timestamp.today() - row["LAST_MOVEMENT_DATE"]).days if pd.notna(row["LAST_MOVEMENT_DATE"]) else 9999
    if days > 180: return "Slow Mover"
    if days > 90:  return "Watch"
    return "Active"

%%time
inv["risk_class"] = inv.apply(classify_material, axis=1)
print(inv["risk_class"].value_counts())

In [ ]:
# -- 3. Vectorized alternative (always prefer over .apply when possible) --
%%time
# Same logic using vectorized ops — significantly faster on large DataFrames
today = pd.Timestamp.today()
days_since = (today - inv["LAST_MOVEMENT_DATE"]).dt.days.fillna(9999)

inv["risk_class_vec"] = np.select(
    condlist=[
        inv["LABST"] == 0,
        days_since > 180,
        days_since > 90,
    ],
    choicelist=["Zero Stock", "Slow Mover", "Watch"],
    default="Active"
)

# Verify equivalence
assert (inv["risk_class"] == inv["risk_class_vec"]).all()
print("Vectorized approach produces identical results and is faster")
print(inv["risk_class_vec"].value_counts())

In [ ]:
# -- 4. Lambda: when to use and when NOT to --

# USE lambdas: short one-off transforms in .agg() / .map()
rev_summary = (sales.groupby("VKORG")["NETWR"]
               .agg(total="sum", p90=lambda x: x.quantile(0.9)))
display(rev_summary)

# AVOID lambdas for:
# 1. Complex logic (use named function — testable, readable)
# 2. Operations better handled by vectorized pandas methods
# 3. Anything you will reuse (name it)

# BAD: df["flag"] = df["NETWR"].apply(lambda x: "HIGH" if x > 10000 else "LOW")
# GOOD (vectorized):
sales["value_band"] = np.where(sales["NETWR"] > 10_000, "HIGH", "LOW")
print("\nValue band distribution:")
print(sales["value_band"].value_counts())

In [ ]:
# -- 5. Data quality report function --

def data_quality_report(df: pd.DataFrame, name: str = "DataFrame") -> pd.DataFrame:
    # Generate a column-level data quality summary.
    # Returns: nulls, uniqueness, dtype, sample values.
    report_rows = []
    for col in df.columns:
        null_count = df[col].isna().sum()
        unique_count = df[col].nunique(dropna=True)
        sample = df[col].dropna().head(3).tolist()
        report_rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "null_count": null_count,
            "null_pct": f"{null_count / len(df):.1%}",
            "unique_count": unique_count,
            "cardinality": f"{unique_count / len(df):.1%}",
            "sample_values": sample,
        })
    result = pd.DataFrame(report_rows).set_index("column")
    print(f"Data Quality Report: {name}  ({len(df):,} rows x {len(df.columns)} cols)")
    return result

display(data_quality_report(inv, "materials_inventory"))

In [ ]:
# -- 6. Parameterized inventory aging report function --

def inventory_aging_report(df: pd.DataFrame, plant: Optional[str] = None) -> pd.DataFrame:
    # Returns inventory aging summary for a given plant (or all plants if None).
    # Buckets: <30d, 30-90d, 90-180d, >180d, Unknown
    # Includes: material count, total stock qty, total inventory value per bucket.
    data = df.copy()
    if plant is not None:
        data = data[data["WERKS"].astype(str).str.zfill(4) == str(plant).zfill(4)]
        if data.empty:
            raise ValueError(f"No records found for plant {plant!r}")

    today = pd.Timestamp.today().normalize()
    data["days_since_mvmt"] = (today - data["LAST_MOVEMENT_DATE"]).dt.days

    bucket_labels = ["< 30 days", "30-90 days", "90-180 days", "> 180 days", "Unknown"]
    bucket_cats   = pd.CategoricalDtype(categories=bucket_labels, ordered=True)

    data["aging_bucket"] = pd.cut(
        data["days_since_mvmt"],
        bins=[-1, 29, 89, 179, float("inf")],
        labels=["< 30 days", "30-90 days", "90-180 days", "> 180 days"]
    ).cat.add_categories("Unknown").fillna("Unknown").astype(bucket_cats)

    data["inv_value"] = data["LABST"] * data["STPRS"]

    summary = (
        data.groupby("aging_bucket", observed=False)
        .agg(
            n_materials=("MATNR", "nunique"),
            total_stock_qty=("LABST", "sum"),
            total_inv_value=("inv_value", "sum"),
        )
        .reset_index()
    )
    summary["inv_value_pct"] = (summary["total_inv_value"] / summary["total_inv_value"].sum() * 100).round(1)

    label = f"Plant {plant}" if plant else "All Plants"
    print(f"\n-- Inventory Aging Report: {label} --")
    return summary

# Demo: all plants
display(inventory_aging_report(inv))

# Demo: specific plant (use first plant in dataset)
plant_sample = inv["WERKS"].astype(str).iloc[0]
display(inventory_aging_report(inv, plant=plant_sample))

---
## Daily Prompt

**Write a function `inventory_aging_report(df, plant)` that takes the inventory DataFrame and a plant code, and returns a summary of stock by aging bucket: `<30d`, `30-90d`, `90-180d`, `>180d` based on `LAST_MOVEMENT_DATE`.**

The function above is a starter. Extend it with:
1. A percentage column for both `total_stock_qty` and `total_inv_value`
2. A `zero_stock_count` column (materials with `LABST == 0` per bucket)
3. A printed summary line: *"Plant XXXX has $Y of inventory idle for >180 days"*

```python
# YOUR SOLUTION
def inventory_aging_report_v2(df: pd.DataFrame, plant: str = None) -> pd.DataFrame:
    # YOUR SOLUTION: extend the function above
    pass

for p in inv["WERKS"].unique():
    report = inventory_aging_report_v2(inv, plant=p)
    display(report)
```

> **Hint:** `zero_stock_count = ("LABST", lambda x: (x == 0).sum())` in `.agg()`.
> For the summary line: `idle = summary.loc[summary["aging_bucket"] == "> 180 days", "total_inv_value"].iloc[0]`